# Четвертая лабораторная работа

## Установка и импорт необходимых библиотек

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Загрузка набора данных

In [ ]:
df = pd.read_csv('cars_moldova_clean.csv', delimiter = ',')
df

## Подготовка данных для логистической регрессии

Для выполнения логистической регрессии нам необходимо:
1. Подготовить целевую переменную (бинарная классификация)
2. Подготовить признаки
3. Разделить данные на обучающую и тестовую выборки

### Создание целевой переменной

Создадим бинарную переменную на основе цены автомобиля. Если цена выше медианной, присвоим 1, иначе 0.

In [ ]:
median_price = df['price'].median()
df['high_price'] = (df['price'] > median_price).astype(int)
print(f'Медианная цена: {median_price}')
print('\nРаспределение классов:')
print(df['high_price'].value_counts(normalize=True))

### Подготовка признаков

Выберем числовые признаки для нашей модели и проведем их нормализацию

In [ ]:
numeric_features = ['year', 'mileage', 'engine_volume']
X = df[numeric_features].copy()

X = (X - X.mean()) / X.std()

y = df['high_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Размеры выборок:')
print(f'Обучающая: {X_train.shape}')
print(f'Тестовая: {X_test.shape}')

## Обучение модели логистической регрессии

Создадим и обучим модель логистической регрессии на наших данных

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Точность модели на тестовой выборке: {accuracy:.4f}')

print('\nОтчет о классификации:')
print(classification_report(y_test, y_pred))

## Визуализация результатов

Создадим матрицу ошибок для визуальной оценки качества классификации

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Матрица ошибок')
plt.xlabel('Предсказанные значения')
plt.ylabel('Фактические значения')
plt.show()

print('\nКоэффициенты модели:')
for feature, coef in zip(numeric_features, model.coef_[0]):
    print(f'{feature}: {coef:.4f}')

### Интерпретация результатов

Положительные коэффициенты указывают на то, что увеличение значения соответствующего признака увеличивает вероятность того, что автомобиль будет иметь высокую цену. Отрицательные коэффициенты, наоборот, указывают на уменьшение этой вероятности.

Матрица ошибок показывает:
- True Negatives (верхний левый): правильно предсказанные недорогие автомобили
- False Positives (верхний правый): недорогие автомобили, ошибочно предсказанные как дорогие
- False Negatives (нижний левый): дорогие автомобили, ошибочно предсказанные как недорогие
- True Positives (нижний правый): правильно предсказанные дорогие автомобили

# Контрольные вопросы

Вопрос 1: 
**Что такое логистическая регрессия?**
- Ответ: Логистическая регрессия - это метод классификации, который предсказывает вероятность принадлежности объекта к определенному классу. Несмотря на название "регрессия", это метод решения задач классификации.

Вопрос 2:
**В чем отличие логистической регрессии от линейной?**
- Ответ: Линейная регрессия предсказывает непрерывные значения, а логистическая регрессия предсказывает вероятности принадлежности к классам (0 или 1). Логистическая регрессия использует сигмоидную функцию для преобразования линейного предсказания в вероятность.

Вопрос 3:
**Какие метрики используются для оценки качества логистической регрессии?**
- Ответ: Основные метрики: accuracy (точность), precision (точность), recall (полнота), F1-score (F1-мера), ROC-AUC. Также используется матрица ошибок (confusion matrix).

Вопрос 4:
**Что показывают коэффициенты логистической регрессии?**
- Ответ: Коэффициенты показывают влияние каждого признака на вероятность принадлежности к классу. Положительный коэффициент увеличивает вероятность, отрицательный - уменьшает.

In [ ]:
_EPS_ = 1e-6

class LogisticRegression_Custom:
    def __init__(self, learning_rate=0.5, epochs=100, threshold=0.5, batch_size=1000, random_state=42):
        self.learning_rate = learning_rate/2
        self.epochs = epochs
        self.threshold = threshold
        self.batch_size = batch_size
        self.random_state = random_state
        self.weights = None
        self.bias = None
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]
    
    def forward(self, X):        
        return self.sigmoid(self.weights.dot(X.T))
    
    def to_class(self, logit):
        return (logit >= self.threshold) * 1
    
    def fit(self, X, y):
        X = self.add_bias(X)
        n_samples, n_features = X.shape
        
        if self.weights is None:
            np.random.seed(self.random_state)
            self.weights = np.random.randn(n_features)
        
        for _ in range(self.epochs):
            yhat = self.forward(X)
            gradient = X.T.dot(yhat - y) / n_samples
            self.weights -= self.learning_rate * gradient
            
    def predict(self, X):
        yhat = self.forward(self.add_bias(X))
        return self.to_class(yhat)
    
    def score(self, X, y):
        yhat = self.predict(X)
        return sum((yhat==y)*1)/y.size